# Week 43, Day 2: Queues and deques

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w043d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

line = deque(["Ada", "Bo"])
line.append("Chi")              # joins at the back
served = line.popleft()         # leaves from the front
line.appendleft("Dee")          # jumps the queue
line.rotate(1)                  # the back person moves to the front
print(served, list(line), line[0], len(line))

In [ ]:
import time
from collections import deque

def empty_list(n):
    q = list(range(n))
    while q:
        q.pop(0)

def empty_deque(n):
    q = deque(range(n))
    while q:
        q.popleft()

for n in [20_000, 40_000, 80_000]:
    t0 = time.perf_counter(); empty_list(n)
    t1 = time.perf_counter(); empty_deque(n)
    t2 = time.perf_counter()
    print(f"n={n:>6}: list.pop(0) {1000 * (t1 - t0):7.1f} ms   deque.popleft() {1000 * (t2 - t1):5.1f} ms")

In [ ]:
from collections import deque

def round_robin(jobs, slice_ms):
    """jobs: (name, ms needed). Returns (name, finish time) in the order they finish."""
    queue = deque(jobs)
    clock, finished = 0, []
    while queue:
        name, left = queue.popleft()
        used = min(slice_ms, left)
        clock += used
        if left > used:
            queue.append((name, left - used))      # back of the queue
        else:
            finished.append((name, clock))
    return finished

print(round_robin([("edit", 30), ("music", 10), ("backup", 50)], 20))

In [ ]:
from collections import deque

recent = deque(maxlen=3)
for page in ["home", "news", "sport", "weather", "news"]:
    recent.append(page)
    print(list(recent))

In [ ]:
from collections import deque

maze = [
    "S.#.....",
    ".##.###.",
    "......#.",
    ".####.#E",
]

def shortest(maze):
    rows, cols = len(maze), len(maze[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if maze[r][c] == "S")
    dist = {start: 0}
    queue = deque([start])
    while queue:
        r, c = queue.popleft()
        if maze[r][c] == "E":
            return dist[(r, c)]
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1           # recorded when first seen: that's the shortest
                queue.append((nr, nc))
    return -1

print(shortest(maze))

> **Key idea.** A queue promises first in, first out: enqueue at the back, dequeue from the front, both O(1). A list's `pop(0)` is O(n), so use linked nodes or, in Python, `collections.deque`, which is O(1) at both ends (but O(n) in the middle) and can be bounded with `maxlen`. Deques suit turn-taking, recent history and time windows. Breadth-first search uses a queue to visit squares in order of distance, so the first time it reaches a place is by a shortest route; mark squares when you enqueue them. Swap the queue for a stack and the same loop becomes depth-first.

### Exercise: A linked queue

Write `LinkedQueue` on `Node`s with a head (front), a tail (back) and a size: `enqueue`, `dequeue` and `peek` (both raising `IndexError("dequeue from empty queue")` and `IndexError("peek at empty queue")`), `__len__`, `__bool__`, `__iter__` (front to back, without `yield`) and `__repr__` like `LinkedQueue([1, 2])` (front first). Every operation O(1) except iterating.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

class LinkedQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queue():\n    \"first in, first out\"\n    q = LinkedQueue()\n    assert not q and repr(q) == \"LinkedQueue([])\"\n    for x in \"abc\":\n        q.enqueue(x)\n    assert repr(q) == \"LinkedQueue(['a', 'b', 'c'])\" and list(q) == [\"a\", \"b\", \"c\"] and len(q) == 3 and q.peek() == \"a\"\n    assert q.dequeue() == \"a\" and q.dequeue() == \"b\"\n    q.enqueue(\"d\")\n    assert list(q) == [\"c\", \"d\"] and q.dequeue() == \"c\" and q.dequeue() == \"d\" and not q\n    q.enqueue(\"e\")\n    assert list(q) == [\"e\"], \"after emptying, the tail must be reset\"\n    q.dequeue()\n    for name, message in [(\"dequeue\", \"dequeue from empty queue\"), (\"peek\", \"peek at empty queue\")]:\n        try:\n            getattr(q, name)()\n            assert False\n        except IndexError as err:\n            assert str(err) == message\ndef test_fast():\n    \"O(1)\"\n    q = LinkedQueue()\n    for i in range(100_000):\n        q.enqueue(i)\n    total = 0\n    while q:\n        total += q.dequeue()\n    assert total == sum(range(100_000)) and \"yield\" not in source")

### Exercise: Taking turns

Write `round_robin(jobs, slice_ms)` with a `deque`: `jobs` is a list of `(name, ms_needed)`, all starting at time 0. The front job runs for `slice_ms` milliseconds or until it's finished, whichever is sooner; an unfinished job goes to the back. Return a list of `(name, finish_time)` in finishing order. A `slice_ms` below 1 raises `ValueError`.

In [ ]:
from collections import deque

def round_robin(jobs, slice_ms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_robin():\n    \"turns\"\n    assert round_robin([(\"edit\", 30), (\"music\", 10), (\"backup\", 50)], 20) == [(\"music\", 30), (\"edit\", 60), (\"backup\", 90)]\n    assert round_robin([(\"a\", 5), (\"b\", 5)], 100) == [(\"a\", 5), (\"b\", 10)] and round_robin([], 10) == []\n    assert round_robin([(\"x\", 1)] * 3, 1) == [(\"x\", 1), (\"x\", 2), (\"x\", 3)]\n    try:\n        round_robin([(\"a\", 1)], 0)\n        assert False\n    except ValueError:\n        pass\n    assert \"deque\" in source")

### Exercise: Requests in the last minute

Write `RateCounter(window)` with `hit(t)`: record a request at time `t` (in seconds; each call's `t` is at least the previous one's) and return how many requests happened in the last `window` seconds, counting those at times `t - window + 1` to `t`. Keep the times in a `deque` and drop old ones from the front, so each call is O(1) amortised.

In [ ]:
from collections import deque

class RateCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rate():\n    \"a time window\"\n    r = RateCounter(60)\n    assert [r.hit(t) for t in [1, 10, 59, 60, 61, 70, 200]] == [1, 2, 3, 4, 4, 4, 1]\n    r = RateCounter(1)\n    assert [r.hit(t) for t in [5, 5, 6]] == [1, 2, 1]\ndef test_fast():\n    \"many hits\"\n    r = RateCounter(1000)\n    for t in range(200_000):\n        n = r.hit(t)\n    assert n == 1000 and len(r._times) <= 1000")

### Exercise: The last lines

Write `last_lines(text, n)`: the last `n` lines of `text` (split with `splitlines()`), as a list, using a `deque` with `maxlen` so that only `n` lines are ever kept while reading. `n` of 0 gives `[]`.

In [ ]:
from collections import deque

def last_lines(text, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tail():\n    \"the tail\"\n    text = \"one\\ntwo\\nthree\\nfour\"\n    assert last_lines(text, 2) == [\"three\", \"four\"] and last_lines(text, 10) == [\"one\", \"two\", \"three\", \"four\"] and last_lines(text, 0) == [] and last_lines(\"\", 3) == []\n    assert \"maxlen\" in source")

### Exercise: Shortest route through a maze

Write `route_length(maze)`: `maze` is a list of strings with `S` (start), `E` (exit), `#` (wall) and `.` (open). Moving up, down, left or right one square is one step. Return the fewest steps from `S` to `E`, or `-1` if there's no way. Use breadth-first search with a `deque`, marking squares when they're added. The checks use a 300 by 300 maze.

In [ ]:
from collections import deque

def route_length(maze):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_small():\n    \"small mazes\"\n    assert route_length([\"S.#.....\", \".##.###.\", \"......#.\", \".####.#E\"]) == 14\n    assert route_length([\"SE\"]) == 1 and route_length([\"S#E\"]) == -1 and route_length([\"S\", \".\", \"E\"]) == 2\n    assert route_length([\"S..\", \"##.\", \"E..\"]) == 6\ndef test_big():\n    \"300 by 300\"\n    m = [\".\" * 300 for _ in range(300)]\n    m[0] = \"S\" + m[0][1:]\n    m[299] = m[299][:299] + \"E\"\n    assert route_length(m) == 598")

### Exercise: Level by level

Write `rings(maze, start)`: using BFS from `start` (a `(row, col)` on an open square; walls are `#`), return a list where item `d` is the **number** of squares exactly `d` steps away (item 0 is 1, the start itself), up to the furthest reachable square.

In [ ]:
from collections import deque

def rings(maze, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rings():\n    \"rings\"\n    assert rings([\"...\", \"...\", \"...\"], (1, 1)) == [1, 4, 4] and rings([\"...\", \"...\", \"...\"], (0, 0)) == [1, 2, 3, 2, 1]\n    assert rings([\".#.\", \".#.\", \"...\"], (0, 0)) == [1, 1, 1, 1, 1, 1, 1] and rings([\".\"], (0, 0)) == [1]")

### Bug hunt: The wrong end

This "queue" takes from the **back** with `pop()`, so the last customer to arrive is served first: this prints `['Chi', 'Bo', 'Ada']`. Fix it so customers are served in the order they arrived, `['Ada', 'Bo', 'Chi']`.

In [ ]:
from collections import deque

line = deque()
for name in ["Ada", "Bo", "Chi"]:
    line.append(name)
served = []
while line:
    served.append(line.pop())
print(served)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"first in, first out\"\n    expect_output(\"['Ada', 'Bo', 'Chi']\")")

### Bug hunt: Seen too late

This BFS marks a square as seen only when it's taken **out** of the queue, so squares are added many times over before they're processed: on this open 4 by 4 grid it adds 49 squares to the queue. Mark squares when they're **added**, so each is added once and it prints `16 16`.

In [ ]:
from collections import deque

def explore(rows, cols):
    seen = set()
    queue = deque([(0, 0)])
    added = 1
    while queue:
        r, c = queue.popleft()
        if (r, c) in seen:
            continue
        seen.add((r, c))
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen:
                queue.append((nr, nc))
                added += 1
    return len(seen), added

print(*explore(4, 4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mark on adding\"\n    expect_output(\"16 16\")")

### Bug hunt: A history that forgets everything

This makes a **new** bounded deque inside the loop, so each one holds only the latest page: it prints `['news']`. Make one deque before the loop, so it prints the last three pages, `['sport', 'weather', 'news']`.

In [ ]:
from collections import deque

for page in ["home", "news", "sport", "weather", "news"]:
    recent = deque(maxlen=3)
    recent.append(page)
print(list(recent))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one deque\"\n    expect_output(\"['sport', 'weather', 'news']\")")

### Puzzle: Breadth-first search

Arrange the lines into the core of a BFS that records each square's distance. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from collections import deque

d = deque([1, 2, 3, 4])
d.rotate(1)
a = list(d)
d.rotate(-2)
b = list(d)
e = deque([1, 2], maxlen=2)
e.append(3)
c = list(e)

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # the cost of list.pop(0) on a list of n items: "O(1)" or "O(n)"
t = ___   # which search finds shortest routes in steps: "breadth-first" or "depth-first"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [4, 1, 2, 3] and q == [2, 3, 4, 1] and r == [2, 3], \"p, q and r\"\n    assert s == \"O(n)\" and t == \"breadth-first\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: spreading rot

A crate of oranges is a grid of `"."` (empty), `"o"` (fresh) and `"x"` (rotten). Every minute, each rotten orange makes the fresh oranges next to it (up, down, left, right) rotten. Write `minutes_to_rot(grid)`: how many minutes until no fresh orange is left, or `-1` if some can never rot. Start the BFS from **every** rotten orange at once (put them all in the queue at distance 0); the answer is the largest distance any fresh orange is reached at. A grid with no fresh oranges needs 0 minutes. The checks use a 300 by 300 crate.

In [ ]:
from collections import deque

def minutes_to_rot(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(grid):\n    g = [list(r) for r in grid]\n    minutes = 0\n    while True:\n        new = [(r, c) for r in range(len(g)) for c in range(len(g[r])) if g[r][c] == \"o\"\n               and any(0 <= r + dr < len(g) and 0 <= c + dc < len(g[r + dr]) and g[r + dr][c + dc] == \"x\" for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)])]\n        if not new:\n            break\n        for r, c in new:\n            g[r][c] = \"x\"\n        minutes += 1\n    return -1 if any(\"o\" in row for row in g) else minutes\ndef test_small():\n    \"small crates\"\n    assert minutes_to_rot([\"xo.\", \"oo.\", \".oo\"]) == 4 and minutes_to_rot([\"xo.\", \".oo\", \"o.o\"]) == -1 and minutes_to_rot([\".x\"]) == 0 and minutes_to_rot([\"...\"]) == 0\n    rng = random.Random(5)\n    for _ in range(200):\n        g = [\"\".join(rng.choice(\".oox\") for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(1, 6))]\n        width = len(g[0])\n        g = [row.ljust(width, \".\")[:width] for row in g]\n        assert minutes_to_rot(g) == slow(g), g\ndef test_big():\n    \"300 by 300\"\n    g = [\"o\" * 300 for _ in range(300)]\n    g[0] = \"x\" + g[0][1:]\n    assert minutes_to_rot(g) == 598")

## Warm-up

A deque ("double-ended queue", said "deck") adds and removes at **both** ends in O(1). Used with `append` and `popleft` only, it's a plain queue: first in, first out.

## A queue is a promise too

| Operation | Does | Cost (promised) |
|---|---|---|
| `enqueue(x)` | add `x` at the back | O(1) |
| `dequeue()` | remove and return the front item (error if empty) | O(1) |
| `peek()` | the front item, without removing it | O(1) |
| `len(q)` | how many are waiting | O(1) |

A Python list **can** behave like a queue (`append` at the back, `pop(0)` from the front), but `pop(0)` shifts every remaining item (Week 41): O(n) per dequeue, O(n²) to empty a queue of n items. Three ways to do it properly:

- **Linked nodes** with a head and a tail (Week 42): enqueue at the tail, dequeue at the head, both O(1).
- **Two stacks** (Week 41's challenge): O(1) amortised.
- **`collections.deque`**: the standard answer in Python.

Doubling n roughly quadruples the list's time and doubles the deque's: O(n²) against O(n).

## collections.deque

CPython's `deque` is a doubly linked list of **blocks**, each holding many items: it has a linked list's O(1) ends, and much of an array's memory efficiency, because items sit side by side within a block.

| Operation | Cost |
|---|---|
| `append(x)`, `appendleft(x)`, `pop()`, `popleft()` | O(1) |
| `extend(xs)`, `extendleft(xs)` | O(k) for k items (`extendleft` reverses their order) |
| `rotate(k)` | O(k): moves k items from the back to the front (or the other way for negative k) |
| `d[0]`, `d[-1]` | O(1) |
| `d[i]` in the middle, `insert`, `remove`, `x in d` | O(n) |
| `deque(maxlen=n)` | a bounded deque: adding at one end drops an item from the other when full |

So a deque is the right tool when you work at the **ends**, and the wrong one when you index into the middle; that's still a list's job.

## Turns, windows and history

Three everyday uses.

**Taking turns** (round robin): serve the front, and if it isn't finished, send it to the back. Computers share a processor between programs this way.

**Recent history**: a deque with `maxlen` keeps only the last n items, throwing out the oldest automatically, which is just what "the last 5 searches" or "the last 100 log lines" needs:

**Time windows**: to count events in the last 60 seconds, keep their times in a deque; when a new one arrives, drop times from the front that are too old. Each time is added once and removed once, so this is O(1) amortised per event: a sliding window (Week 41) on a stream that never ends.

## Breadth-first search

Day 1 searched a grid with a stack, going as deep as possible down one path before trying another: **depth-first**. Use a **queue** instead, and the search spreads out in rings: first every square one step away, then every square two steps away, and so on. That's **breadth-first search** (BFS), and its great property is that the first time it reaches a square, it has found a **shortest** route there.

Why is the first route found the shortest? The queue always holds squares in order of distance: everything at distance d comes out before anything at distance d + 1, because d + 1 squares are only added while d squares are being processed. So squares are reached in order of their distance from the start. (BFS is credited to Edward F. Moore, who used it in 1959 to find the shortest path out of a maze; you'll meet it again on graphs later in this phase.)

Two details matter:

- **Mark a square when you add it to the queue**, not when you take it out. Otherwise the same square can be added many times before it's processed.
- BFS takes O(squares) time and space: each square is added and removed at most once.

## Stack or queue?

The flood fill from Day 1 and the BFS above are the **same loop**: take a square, process it, add its neighbours. The only difference is which end the next square comes from:

| Container | Takes from | Search | Good for |
|---|---|---|---|
| stack (`pop()`) | the newest | depth-first | exploring everything, backtracking, low memory on wide searches |
| queue (`popleft()`) | the oldest | breadth-first | shortest paths in steps, nearest first, levels |

## Practice

## Quiz

## Daily challenge

## Before you go

Take Day 1's island counter, swap its stack for a deque used as a queue, and check it still gives the same sizes. Then explain why it does, even though the squares are visited in a different order. Tomorrow: monotonic stacks, a small twist on the stack that answers "what's the next bigger number?" for every item in one pass.